In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -r /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 129.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.5/212.5 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 135.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 kB 3.9 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24988 sha256=72374cacb3bf23b08e45d4e5ebc8ead81b753e302429972792985f2b2733acc9
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef54687

In [3]:
!pip install -q transformers datasets sentencepiece accelerate evaluate rouge-score

In [4]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
)
import evaluate
from tqdm import tqdm

In [5]:
PROJECT_ROOT = Path(
    "/content/drive/MyDrive/ml_projects/NLP_Text_Summarization"
)

PROCESSED_DATA_DIR = PROJECT_ROOT / "artifacts/data/processed"
TOKENIZER_DIR = PROJECT_ROOT / "artifacts/tokenizer"
BASELINE_OUTPUT_DIR = PROJECT_ROOT / "artifacts/baseline_outputs"

BASELINE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(PROCESSED_DATA_DIR)
print(TOKENIZER_DIR)
print(BASELINE_OUTPUT_DIR)

/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/data/processed
/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/tokenizer
/content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/baseline_outputs


In [6]:
train_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_train"
)
validation_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_validation"
)
test_dataset = load_from_disk(
    PROCESSED_DATA_DIR / "tokenized_test"
)
print(train_dataset)
print(validation_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 14731
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 818
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 819
})


In [7]:
MODEL_NAME = "google/pegasus-large"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/88.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.28G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/680 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
PegasusForConditionalGeneration LOAD REPORT from: google/pegasus-large
Key                                  | Status  | 
-------------------------------------+---------+-
model.decoder.embed_positions.weight | MISSING | 
model.encoder.embed_positions.weight | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [8]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print("Using device:", device)
model.to(device)

Using device: cpu


PegasusForConditionalGeneration(
  (model): PegasusModel(
    (shared): Embedding(96103, 1024, padding_idx=0)
    (encoder): PegasusEncoder(
      (embed_tokens): Embedding(96103, 1024, padding_idx=0)
      (embed_positions): PegasusSinusoidalPositionalEmbedding(1024, 1024)
      (layers): ModuleList(
        (0-15): 16 x PegasusEncoderLayer(
          (self_attn): PegasusAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): ReLU()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (final_layer_no

In [9]:
rouge = evaluate.load("rouge")

In [10]:
def generate_summary(dialogue_text):

    dialogue_text = "summarize: " + dialogue_text

    inputs = tokenizer(
        dialogue_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding="longest",
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }
    summary_ids = model.generate(
    **inputs,

    max_length=64,
    min_length=8,

    num_beams=8,

    repetition_penalty=2.5,

    no_repeat_ngram_size=3,

    length_penalty=1.0,

    early_stopping=True,
)
    generated_summary = tokenizer.decode(
        summary_ids[0],
        skip_special_tokens=True,
    )
    return generated_summary

In [11]:
from datasets import load_dataset
raw_dataset = load_dataset("knkarthick/samsum")
sample_dialogue = raw_dataset["test"][0]["dialogue"]
reference_summary = raw_dataset["test"][0]["summary"]
print("Dialogue:\n")
print(sample_dialogue)

README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14731 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/818 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/819 [00:00<?, ? examples/s]

Dialogue:

Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye


In [12]:
baseline_summary = generate_summary(sample_dialogue)

print("Generated Summary:\n")
print(baseline_summary)
print("Reference Summary:\n")
print(reference_summary)

Generated Summary:

durable previous walking well Slowly informed well aren well married free Ser free smartphone articles offers
Reference Summary:

Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.


In [13]:
EVAL_SAMPLES = 50
generated_summaries = []
reference_summaries = []

In [14]:
for sample in tqdm(raw_dataset["test"].select(range(EVAL_SAMPLES))):
    dialogue = sample["dialogue"]
    reference = sample["summary"]

    prediction = generate_summary(dialogue)
    generated_summaries.append(prediction)
    reference_summaries.append(reference)

100%|██████████| 50/50 [11:46<00:00, 14.12s/it]


In [15]:
results = rouge.compute(
    predictions=generated_summaries,
    references=reference_summaries,
)

In [16]:
results

{'rouge1': np.float64(0.013409118667357379),
 'rouge2': np.float64(0.0),
 'rougeL': np.float64(0.012717370923641476),
 'rougeLsum': np.float64(0.012616446569396748)}

In [17]:
results_df = pd.DataFrame([results])
results_df

,rouge1,rouge2,rougeL,rougeLsum
0,0.013409,0.0,0.012717,0.012616


In [18]:
results_path = BASELINE_OUTPUT_DIR / "baseline_rouge_scores.csv"
results_df.to_csv(results_path, index=False)
print(f"Saved baseline metrics to: {results_path}")

Saved baseline metrics to: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/baseline_outputs/baseline_rouge_scores.csv


In [19]:
prediction_df = pd.DataFrame({
    "reference_summary": reference_summaries,
    "generated_summary": generated_summaries,
})
prediction_output_path = (
    BASELINE_OUTPUT_DIR / "baseline_predictions.csv"
)
prediction_df.to_csv(
    prediction_output_path,
    index=False,
)
print(f"Saved predictions to: {prediction_output_path}")

Saved predictions to: /content/drive/MyDrive/ml_projects/NLP_Text_Summarization/artifacts/baseline_outputs/baseline_predictions.csv


In [20]:
real_world_dialogues = [

    """
    bro kal aa rha h kya 😭
    nhi yrr internals h
    fir weekend?
    dekhte h lol
    """,


    """
    User: hello support team my payment got deducted twice
    Agent: sorry for inconvenience please share transaction id
    User: TXN45872
    Agent: refund will be processed in 5 working days
    """,


    """
    Mike: dude GPU crashed again
    Alex: CUDA out of memory?
    Mike: yes while training LoRA
    Alex: reduce batch size maybe
    """,
]

In [21]:
for idx, conversation in enumerate(real_world_dialogues):

    print("=" * 80)
    print(f"Conversation {idx+1}")
    print("=" * 80)
    print("\nInput Conversation:\n")
    print(conversation)
    generated = generate_summary(conversation)
    print("\nGenerated Summary:\n")
    print(generated)

Conversation 1

Input Conversation:


    bro kal aa rha h kya 😭
    nhi yrr internals h
    fir weekend?
    dekhte h lol
    

Generated Summary:

always way reason Days – quarterback very now Do average well
Conversation 2

Input Conversation:


    User: hello support team my payment got deducted twice
    Agent: sorry for inconvenience please share transaction id
    User: TXN45872
    Agent: refund will be processed in 5 working days
    

Generated Summary:

mem flesh should Serviceve day Aug Children however through informed now socks leverage now socks 44 now socks
Conversation 3

Input Conversation:


    Mike: dude GPU crashed again
    Alex: CUDA out of memory?
    Mike: yes while training LoRA
    Alex: reduce batch size maybe
    

Generated Summary:

me Woburn institutions first world In good provide development very need during her own information breakthrough very years Scale first lot wayWere – little growth her those best well


In [22]:
print("=" * 60)
print("Baseline Drift Analysis Observations")
print("=" * 60)

print("""
Observe:
- coherence quality,
- slang understanding,
- noisy conversation handling,
- domain adaptation limitations,
- semantic compression robustness.

These outputs establish the drift-aware baseline
before LoRA adaptation.
""")

Baseline Drift Analysis Observations

Observe:
- coherence quality,
- slang understanding,
- noisy conversation handling,
- domain adaptation limitations,
- semantic compression robustness.

These outputs establish the drift-aware baseline
before LoRA adaptation.



In [23]:
print("=" * 60)
print("Notebook 04 Completed Successfully")
print("=" * 60)

print("Baseline PEGASUS evaluation completed.")
print("ROUGE metrics generated.")
print("Real-world drift conversations tested.")
print("Baseline outputs saved successfully.")

Notebook 04 Completed Successfully
Baseline PEGASUS evaluation completed.
ROUGE metrics generated.
Real-world drift conversations tested.
Baseline outputs saved successfully.
